# LeetCode #46 — Permutations: Backtracking với mảng đánh dấu `used`

**Mức độ:** Medium · **Chủ đề:** Array, Backtracking

Skeleton trên LeetCode:

```python
class Solution:
    def permute(self, nums: List[int]) -> List[List[int]]:
```

Ở bài #39 / #40 ta sinh **tổ hợp** (thứ tự không quan trọng) bằng chỉ số `start`. **Hoán vị** thì khác: thứ tự **quan trọng** (`[1, 2]` khác `[2, 1]`), và ở mỗi vị trí ta được chọn **bất kỳ** phần tử nào **chưa dùng** — kể cả phần tử đứng trước.

Mục tiêu của notebook:

1. Phân biệt **tổ hợp** và **hoán vị** trong backtracking.
2. Dùng mảng `used` để đánh dấu phần tử đã chọn — và nhớ **gỡ đánh dấu** khi quay lui.
3. Cách thứ hai: **đổi chỗ tại chỗ (swap)**.
4. Đếm số kết quả: `n!`.

# 1. Bài toán là gì?

Cho mảng `nums` gồm các số nguyên **khác nhau**. Trả về **mọi hoán vị**. Thứ tự kết quả tùy ý.

**Constraints (theo đề):**

- `1 <= nums.length <= 6`
- `-10 <= nums[i] <= 10`
- Mọi phần tử **khác nhau**.

---

## Ví dụ

```text
[1, 2, 3]  ->  [[1,2,3], [1,3,2], [2,1,3], [2,3,1], [3,1,2], [3,2,1]]
[0, 1]     ->  [[0,1], [1,0]]
[1]        ->  [[1]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đệ quy (recursion)
2. Quay lui (backtracking)
3. Hoán vị, tổ hợp, tập con và thứ tự từ điển

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [1]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [2]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [3]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [4]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [5]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [6]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [7]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [8]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [9]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [10]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Hoán vị, tổ hợp, tập con và thứ tự từ điển

### Đếm

| Đối tượng | Ý nghĩa | Số lượng |
|---|---|---|
| Hoán vị của `n` phần tử | sắp **thứ tự** tất cả | `n!` |
| Chỉnh hợp chập `k` | chọn `k` phần tử **có thứ tự** | `n! / (n−k)!` |
| Tổ hợp chập `k` | chọn `k` phần tử **không** thứ tự | `C(n, k) = n! / (k! (n−k)!)` |
| Tập con | mỗi phần tử lấy/không | `2^n` |
| Hoán vị có lặp | `n` phần tử, nhóm trùng cỡ `c1, c2, …` | `n! / (c1! c2! …)` |

In [11]:
from math import factorial, comb, perm
from itertools import permutations, combinations, product

print("4! =", factorial(4), "| P(4,2) =", perm(4, 2), "| C(4,2) =", comb(4, 2), "| 2^4 =", 2 ** 4)
print("hoán vị [1,2,3]:", list(permutations([1, 2, 3])))
print("tổ hợp chập 2 của [1,2,3,4]:", list(combinations([1, 2, 3, 4], 2)))
print("hoán vị khác nhau của 'aab':", sorted(set(permutations("aab"))), "=", factorial(3) // factorial(2))
print("tích Descartes {0,1}^2:", list(product([0, 1], repeat=2)))

4! = 24 | P(4,2) = 12 | C(4,2) = 6 | 2^4 = 16
hoán vị [1,2,3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tổ hợp chập 2 của [1,2,3,4]: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
hoán vị khác nhau của 'aab': [('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a')] = 3
tích Descartes {0,1}^2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### Thứ tự từ điển (lexicographic order)

So hai dãy như so từ trong từ điển: so phần tử **đầu tiên khác nhau**; nếu một dãy là tiền tố của dãy kia, dãy ngắn hơn đứng trước. Python so sánh list/tuple/str đúng theo kiểu này.

Liệt kê hoán vị của `[1, 2, 3]` theo thứ tự từ điển: `123, 132, 213, 231, 312, 321`.

**Quan sát về hoán vị kế tiếp:** đi từ phải sang, phần **hậu tố giảm dần** dài nhất là hậu tố "đã lớn nhất có thể". Muốn hoán vị lớn hơn kế tiếp, phải tăng phần tử **ngay trước** hậu tố đó lên một chút.

In [12]:
print([1, 2, 3] < [1, 3], "|", "abc" < "abd", "|", [1, 2] < [1, 2, 0], "|", "10" < "9")
for p in permutations([1, 2, 3]):
    i = len(p) - 1
    while i > 0 and p[i - 1] >= p[i]:
        i -= 1
    print(p, "| hậu tố giảm dần dài nhất:", p[i:])

True | True | True | True
(1, 2, 3) | hậu tố giảm dần dài nhất: (3,)
(1, 3, 2) | hậu tố giảm dần dài nhất: (3, 2)
(2, 1, 3) | hậu tố giảm dần dài nhất: (3,)
(2, 3, 1) | hậu tố giảm dần dài nhất: (3, 1)
(3, 1, 2) | hậu tố giảm dần dài nhất: (2,)
(3, 2, 1) | hậu tố giảm dần dài nhất: (3, 2, 1)


### Liên hệ với quay lui

`itertools` sinh sẵn hoán vị/tổ hợp, nhưng phỏng vấn thường yêu cầu **tự sinh** bằng quay lui (để kèm cắt tỉa, bỏ trùng, điều kiện riêng). Kích thước kết quả (`n!`, `C(n,k)`, `2^n`) cho biết độ phức tạp tối thiểu — không thuật toán nào liệt kê nhanh hơn số lời giải.

## 2.4 Áp dụng vào bài này

Sinh mọi **hoán vị** (`n!` kết quả) bằng **quay lui** với mảng `used`: mỗi tầng chọn một phần tử chưa dùng, đi tiếp, rồi bỏ chọn.

# 3. Tổ hợp vs. Hoán vị

| | Tổ hợp (#39, #40, #78) | Hoán vị (#46) |
|---|---|---|
| Thứ tự | không quan trọng | **quan trọng** |
| `[1, 2]` và `[2, 1]` | là **một** | là **hai** |
| Ở mỗi tầng được chọn | phần tử từ `start` trở đi | **mọi** phần tử chưa dùng |
| Kỹ thuật | chỉ số `start` | mảng `used` (hoặc swap) |
| Số kết quả (chọn hết `n`) | 1 | `n!` |

Cây quyết định cho `[1, 2, 3]`:

```text
                      []
          /           |           \
        [1]          [2]          [3]          <- vị trí 0: chọn 1 trong 3
       /   \        /   \        /   \
    [1,2] [1,3]  [2,1] [2,3]  [3,1] [3,2]      <- vị trí 1: chọn 1 trong 2 số còn lại
      |     |      |     |      |     |
   [1,2,3] ...                                  <- vị trí 2: số cuối cùng
```

`3 × 2 × 1 = 6` lá.

# 4. Lời giải: Backtracking với `used`

```python
def backtrack():
    if len(path) == n:
        result.append(path[:])
        return
    for i in range(n):
        if used[i]:
            continue
        used[i] = True;  path.append(nums[i])     # chọn
        backtrack()                                # đi tiếp
        path.pop();      used[i] = False          # bỏ chọn — gỡ CẢ HAI
```

In [13]:
from typing import List

def permute(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path = [], []
    used = [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

print(permute([1, 2, 3]))
print(permute([0, 1]))
print(permute([1]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
[[0, 1], [1, 0]]
[[1]]


**Vì sao dùng mảng `used` thay vì kiểm tra `nums[i] in path`?** `x in path` tốn `O(n)` mỗi lần; `used[i]` là `O(1)`. Hơn nữa `in path` sẽ **sai** khi mảng có phần tử trùng (bài #47).

# 5. Mô phỏng từng bước

In [14]:
def trace_permute(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        indent = "  " * len(path)
        if len(path) == n:
            print(f"{indent}✓ {path}")
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            print(f"{indent}chọn {nums[i]} -> path = {path}, used = {used}")
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

_ = trace_permute([1, 2, 3])

chọn 1 -> path = [1], used = [True, False, False]
  chọn 2 -> path = [1, 2], used = [True, True, False]
    chọn 3 -> path = [1, 2, 3], used = [True, True, True]
      ✓ [1, 2, 3]
  chọn 3 -> path = [1, 3], used = [True, False, True]
    chọn 2 -> path = [1, 3, 2], used = [True, True, True]
      ✓ [1, 3, 2]
chọn 2 -> path = [2], used = [False, True, False]
  chọn 1 -> path = [2, 1], used = [True, True, False]
    chọn 3 -> path = [2, 1, 3], used = [True, True, True]
      ✓ [2, 1, 3]
  chọn 3 -> path = [2, 3], used = [False, True, True]
    chọn 1 -> path = [2, 3, 1], used = [True, True, True]
      ✓ [2, 3, 1]
chọn 3 -> path = [3], used = [False, False, True]
  chọn 1 -> path = [3, 1], used = [True, False, True]
    chọn 2 -> path = [3, 1, 2], used = [True, True, True]
      ✓ [3, 1, 2]
  chọn 2 -> path = [3, 2], used = [False, True, True]
    chọn 1 -> path = [3, 2, 1], used = [True, True, True]
      ✓ [3, 2, 1]


# 6. Cách 2: đổi chỗ tại chỗ (swap)

Ý tưởng: vị trí `0..first-1` đã cố định; với vị trí `first`, lần lượt **đưa mỗi phần tử còn lại** (`nums[first..n-1]`) lên vị trí `first` bằng cách đổi chỗ, rồi đệ quy cho `first + 1`, rồi **đổi lại**.

```python
def backtrack(first):
    if first == n:
        result.append(nums[:])
        return
    for i in range(first, n):
        nums[first], nums[i] = nums[i], nums[first]
        backtrack(first + 1)
        nums[first], nums[i] = nums[i], nums[first]   # hoàn tác
```

Không cần `path` hay `used` — chính `nums` đóng vai trò đó: phần `nums[:first]` là "đã chọn", `nums[first:]` là "chưa dùng".

In [15]:
def permute_swap(nums: List[int]) -> List[List[int]]:
    nums = nums[:]               # không sửa input của người gọi
    n = len(nums)
    result = []

    def backtrack(first: int):
        if first == n:
            result.append(nums[:])
            return
        for i in range(first, n):
            nums[first], nums[i] = nums[i], nums[first]
            backtrack(first + 1)
            nums[first], nums[i] = nums[i], nums[first]

    backtrack(0)
    return result

print(permute_swap([1, 2, 3]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 2, 1], [3, 1, 2]]


Thứ tự kết quả khác bản `used` (`[3, 2, 1]` đứng trước `[3, 1, 2]`), nhưng **tập** kết quả giống nhau. Đề cho phép thứ tự tùy ý.

## Cách 3: `itertools.permutations`

In [16]:
from itertools import permutations

print([list(p) for p in permutations([1, 2, 3])])

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


# 7. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def permute(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        result, path = [], []
        used = [False] * n

        def backtrack():
            if len(path) == n:
                result.append(path[:])
                return
            for i in range(n):
                if used[i]:
                    continue
                used[i] = True
                path.append(nums[i])
                backtrack()
                path.pop()
                used[i] = False

        backtrack()
        return result

In [18]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    assert len(actual) == len({tuple(p) for p in actual}), (name, "có trùng")
    print(f"{name:<14} PASS  ({len(actual)} hoán vị)")


sol = Solution()
check("Ví dụ 1", sol.permute([1, 2, 3]), [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
check("Ví dụ 2", sol.permute([0, 1]), [[0, 1], [1, 0]])
check("Ví dụ 3", sol.permute([1]), [[1]])
check("Có số âm", sol.permute([-1, 0, 1]), [list(p) for p in permutations([-1, 0, 1])])

import random
from math import factorial
random.seed(46)
for _ in range(200):
    nums = random.sample(range(-10, 11), random.randint(1, 6))
    expected = sorted(list(p) for p in permutations(nums))
    assert sorted(sol.permute(nums)) == expected
    assert sorted(permute_swap(nums)) == expected
    assert len(expected) == factorial(len(nums))
print("200 test ngẫu nhiên: khớp itertools, số kết quả = n!")

Ví dụ 1        PASS  (6 hoán vị)
Ví dụ 2        PASS  (2 hoán vị)
Ví dụ 3        PASS  (1 hoán vị)
Có số âm       PASS  (6 hoán vị)


200 test ngẫu nhiên: khớp itertools, số kết quả = n!

# 8. Vì sao đúng? Độ phức tạp?

**Đúng:** ở tầng `k`, `path` có `k` phần tử đôi một khác (theo chỉ số), và vòng `for` thử **mọi** chỉ số chưa dùng. Mỗi hoán vị ứng với đúng một đường đi gốc → lá. Gỡ `used[i] = False` sau khi quay lui đảm bảo chỉ số `i` lại **khả dụng** cho các nhánh khác.

**Độ phức tạp:**

- Số lá: `n!`. Mỗi lá tốn `O(n)` để sao chép `path`.
- Số nút trong cây: `n!/0! + n!/1! + … ≈ e · n!` → vẫn `O(n!)`.

```text
Time:  O(n · n!)
Space: O(n) cho path, used và độ sâu đệ quy (không tính kết quả O(n · n!))
```

Với `n = 6`: `6! = 720` hoán vị — rất nhỏ. Nhưng `n!` tăng cực nhanh: `10! ≈ 3.6 triệu`, `15! ≈ 1.3 * 10^12`.

In [19]:
for n in [3, 6, 8, 10, 12]:
    print(f"{n}! = {factorial(n):,}")

3! = 6
6! = 720
8! = 40,320
10! = 3,628,800
12! = 479,001,600


# 9. Những lỗi dễ mắc

## Lỗi 1: Quên gỡ `used[i] = False`

Sau nhánh đầu tiên, mọi phần tử đều bị đánh dấu "đã dùng" vĩnh viễn → chỉ sinh được **một** hoán vị.

In [20]:
def wrong_no_unmark(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            # thiếu used[i] = False

    backtrack()
    return result

print("Sai :", wrong_no_unmark([1, 2, 3]))

Sai : [[1, 2, 3]]


## Lỗi 2: Dùng chỉ số `start` như bài tổ hợp

Chỉ sinh được **một** thứ tự (tăng dần chỉ số) — tức chỉ sinh tổ hợp, không phải hoán vị.

In [21]:
def wrong_start_index(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path = [], []

    def backtrack(start):
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(start, n):           # SAI cho hoán vị
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return result

print("Sai :", wrong_start_index([1, 2, 3]))

Sai : [[1, 2, 3]]


## Lỗi 3: Lưu `nums` (bản swap) hoặc `path` không sao chép

`result.append(nums)` ở bản swap: mọi phần tử của `result` là **cùng một** list, cuối cùng đều bằng trạng thái cuối của `nums`.

## Lỗi 4: Bản swap quên đổi lại

Không hoàn tác → `nums` bị xáo trộn dần, sinh ra hoán vị trùng và thiếu.

# 10. Mẹo nhận diện pattern

- "Mọi **cách sắp xếp**", "mọi **thứ tự**" → **hoán vị** → backtracking với `used`.
- `n` rất nhỏ (≤ 8–10) là dấu hiệu kết quả có thể tới `n!`.
- Có phần tử trùng → **#47 Permutations II** (sort + điều kiện bỏ trùng với `used`).
- Chỉ cần hoán vị **kế tiếp** → **#31 Next Permutation** (`O(n)`, không sinh hết).
- Chỉ cần hoán vị **thứ k** → **#60 Permutation Sequence** (đếm bằng `factorial`).

## Template

```python
used = [False] * n
def backtrack():
    if len(path) == n: result.append(path[:]); return
    for i in range(n):
        if used[i]: continue
        used[i] = True; path.append(nums[i])
        backtrack()
        path.pop(); used[i] = False
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `[1, 2, 3, 4]`: có bao nhiêu hoán vị bắt đầu bằng `3`? Liệt kê theo thứ tự mà bản `used` sinh ra.

## Bài 2 — Hoán vị của chuỗi

Sinh mọi hoán vị của một chuỗi có các ký tự **khác nhau**, trả về list chuỗi.

```python
string_permutations("abc") -> ["abc", "acb", "bac", "bca", "cab", "cba"]
```

In [22]:
def exercise_2_string_permutations(s: str) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_2_string_permutations("abc")) == ["abc", "acb", "bac", "bca", "cab", "cba"]
# assert exercise_2_string_permutations("z") == ["z"]

## Bài 3 — Điền chỗ trống

In [23]:
def exercise_3_permute(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        # TODO 1: base case khi path đủ n phần tử
        # TODO 2: duyệt mọi i chưa dùng: đánh dấu, thêm, đệ quy, bỏ, gỡ đánh dấu
        pass

    backtrack()
    return result


# assert sorted(exercise_3_permute([1, 2, 3])) == sorted([[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
# assert exercise_3_permute([1]) == [[1]]

## Bài 4 — Biến thể: chỉnh hợp chập `k`

Sinh mọi dãy **có thứ tự** gồm `k` phần tử khác nhau lấy từ `nums` (chỉnh hợp chập `k`).

```python
arrangements([1, 2, 3], 2) -> [[1,2], [1,3], [2,1], [2,3], [3,1], [3,2]]
```

In [24]:
def exercise_4_arrangements(nums: List[int], k: int) -> List[List[int]]:
    # TODO
    pass


# assert sorted(exercise_4_arrangements([1, 2, 3], 2)) == [[1, 2], [1, 3], [2, 1], [2, 3], [3, 1], [3, 2]]
# assert len(exercise_4_arrangements([1, 2, 3, 4], 3)) == 24
# assert exercise_4_arrangements([1, 2], 0) == [[]]

## Bài 5 — Mini challenge: Permutation Sequence (LeetCode #60)

Trả về hoán vị **thứ `k`** (đánh số từ 1) của `[1, 2, ..., n]` theo thứ tự từ điển, **không** sinh hết các hoán vị.

```python
get_permutation(3, 3) -> "213"
get_permutation(4, 9) -> "2314"
get_permutation(3, 1) -> "123"
```

Gợi ý nhanh: có `(n-1)!` hoán vị bắt đầu bằng mỗi số. Chữ số đầu tiên là số thứ `(k-1) // (n-1)!` trong danh sách còn lại.

In [25]:
def exercise_5_get_permutation(n: int, k: int) -> str:
    # TODO
    pass


# assert exercise_5_get_permutation(3, 3) == "213"
# assert exercise_5_get_permutation(4, 9) == "2314"
# assert exercise_5_get_permutation(3, 1) == "123"

# 12. Gợi ý

**Bài 1**
- Mức 1: cố định `3` ở đầu, còn 3 số → `3! = 6` hoán vị.

**Bài 2**
- Mức 1: dùng `permute(list(s))` rồi `"".join`, hoặc viết backtracking trực tiếp với `path` là list ký tự.

**Bài 4**
- Mức 1: giống bài chính, đổi base case thành `len(path) == k`.

**Bài 5**
- Mức 1: `digits = [1..n]`, `k -= 1` (đổi sang đánh số từ 0).
- Mức 2: với mỗi vị trí, `idx, k = divmod(k, factorial(len(digits) - 1))`, lấy `digits.pop(idx)`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
all_perms = permute([1, 2, 3, 4])
starting_with_3 = [p for p in all_perms if p[0] == 3]
print(len(starting_with_3), starting_with_3)

6 [[3, 1, 2, 4], [3, 1, 4, 2], [3, 2, 1, 4], [3, 2, 4, 1], [3, 4, 1, 2], [3, 4, 2, 1]]


## Đáp án Bài 2

In [27]:
def answer_2_string_permutations(s: str) -> List[str]:
    return ["".join(p) for p in permute(list(s))]

assert sorted(answer_2_string_permutations("abc")) == ["abc", "acb", "bac", "bca", "cab", "cba"]
assert answer_2_string_permutations("z") == ["z"]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [28]:
def answer_3_permute(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

assert sorted(answer_3_permute([1, 2, 3])) == sorted([[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
assert answer_3_permute([1]) == [[1]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_arrangements(nums: List[int], k: int) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == k:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

assert sorted(answer_4_arrangements([1, 2, 3], 2)) == [[1, 2], [1, 3], [2, 1], [2, 3], [3, 1], [3, 2]]
assert len(answer_4_arrangements([1, 2, 3, 4], 3)) == 24
assert answer_4_arrangements([1, 2], 0) == [[]]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_get_permutation(n: int, k: int) -> str:
    digits = list(range(1, n + 1))
    k -= 1
    out = []
    while digits:
        block = factorial(len(digits) - 1)
        idx, k = divmod(k, block)
        out.append(str(digits.pop(idx)))
    return "".join(out)

assert answer_5_get_permutation(3, 3) == "213"
assert answer_5_get_permutation(4, 9) == "2314"
assert answer_5_get_permutation(3, 1) == "123"
for n in range(1, 7):
    ordered = ["".join(map(str, p)) for p in permutations(range(1, n + 1))]
    assert all(answer_5_get_permutation(n, k + 1) == ordered[k] for k in range(len(ordered)))
print("Bài 5 PASS (khớp mọi k với n = 1..6)")

Bài 5 PASS (khớp mọi k với n = 1..6)


`itertools.permutations` trên dãy đã sắp xếp sinh đúng thứ tự từ điển, nên dùng nó để kiểm tra.

# 14. Tổng kết

> Hoán vị: ở mỗi vị trí chọn **bất kỳ** phần tử **chưa dùng**. Dùng mảng `used` — đánh dấu khi chọn, **gỡ đánh dấu** khi quay lui. Có `n!` kết quả.

---

## Checklist trước khi nộp

- [ ] Duyệt `for i in range(n)` (không dùng `start`)?
- [ ] Bỏ qua `used[i]`?
- [ ] Gỡ **cả** `path.pop()` **và** `used[i] = False`?
- [ ] Lưu `path[:]`?